In [117]:
# Supervised learning (ML) to be performed on reddit_dead_internet_analysis_2026.csv
# Input features account_age_days, user_karma(Optional), reply_delay_seconds, avg_word_length and contains_links
# Drop bot_probability due to leakage risk and is_bot_flag since it is the target
# Models to use Logistic Regression, Random Forest, SVM, KNN and Gradient Boosting (XGBoost / LightGBM)
# Goal to perform classification using the ml models and the create a table with columns: Alorithm, Accuracy, Precision, Recall, F1-Score, r^2 and Training Time

In [118]:
# ================================
# 1. IMPORT LIBRARIES
# ================================
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, r2_score

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Gradient Boosting (choose one)
from xgboost import XGBClassifier
# from lightgbm import LGBMClassifier  # optional alternative

In [ ]:
# ================================
# 2. LOAD DATASET
# ================================

df = pd.read_csv("SEM 6\DS Lab\reddit_dead_internet_analysis_2026.csv")

In [120]:
# ================================
# 3. FEATURE SELECTION
# ================================
features = ["account_age_days", "user_karma", "reply_delay_seconds", "avg_word_length", "contains_links", "sentiment_score"]
target = "is_bot_flag"
X=df[features]
y=df[target]

In [121]:
# ================================
# 4. TRAIN-TEST SPLIT
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

In [122]:
# ================================
# 5. SCALING (ONLY FOR LR)
# ================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [123]:
# ================================
# 6. MODEL TRAINING FUNCTION
# ================================
results = []

def evaluate_model(name, model, X_tr, X_te):
    start_time = time.time()
    
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    
    end_time = time.time()
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    train_time = end_time - start_time
    
    # 5-fold stratified CV on the training split
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_f1_scores = cross_val_score(model, X_tr, y_train, cv=cv, scoring="f1")
    cv_f1_mean = cv_f1_scores.mean()
    
    results.append([name, acc, prec, rec, f1, cv_f1_mean, r2, train_time])

In [124]:
# ================================
# 7. TRAIN MODELS
# ================================

# Logistic Regression
lr = LogisticRegression(max_iter=1000)
evaluate_model("Logistic Regression", lr, X_train_scaled, X_test_scaled)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
evaluate_model("Random Forest", rf, X_train, X_test)

# Support Vector Machine (SVM)
svm = SVC()
evaluate_model("SVM", svm, X_train_scaled, X_test_scaled)

# K-Nearest Neighbors (KNN)
knn = KNeighborsClassifier(n_neighbors=5)
evaluate_model("KNN", knn, X_train_scaled, X_test_scaled)

# XGBoost
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
evaluate_model("XGBoost", xgb, X_train, X_test)

# OPTIONAL: LightGBM
# lgbm = LGBMClassifier()
# evaluate_model("LightGBM", lgbm, X_train, X_test)

C:\Users\siddi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\training.py:200: UserWarning: [10:49:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\siddi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\training.py:200: UserWarning: [10:49:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\siddi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\training.py:200: UserWarning: [10:49:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } 

In [125]:
# ================================
# 8. RESULTS TABLE
# ================================
results_df = pd.DataFrame(results, columns=[
    "Algorithm",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-Score",
    "CV F1 (5-Fold)",
    "R^2",
    "Training Time (s)"
])

# Sort by F1-score (best metric for classification imbalance)
results_df = results_df.reset_index(drop=True)

In [126]:
# ================================
# 9. DISPLAY RESULTS
# ================================
print("\nModel Comparison Table:\n")
display(results_df)


Model Comparison Table:



,Algorithm,Accuracy,Precision,Recall,F1-Score,CV F1 (5-Fold),R^2,Training Time (s)
0,Logistic Regression,0.984,1.000000,0.962963,0.981132,0.978924,0.934794,0.010063
1,Random Forest,1.000,1.000000,1.000000,1.000000,0.997015,1.000000,0.329795
2,SVM,0.984,1.000000,0.962963,0.981132,0.972302,0.934794,0.002996
3,KNN,0.960,0.980392,0.925926,0.952381,0.959866,0.836985,0.003362
4,XGBoost,1.000,1.000000,1.000000,1.000000,0.994118,1.000000,0.038343
